In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt 
import math

torch.set_default_dtype(torch.float64)


In [3]:

def U0(z, t):
    z = torch.as_tensor(z, dtype=torch.float64)
    t = torch.as_tensor(t, dtype=torch.float64)
    sqrt_4t = torch.sqrt(4 * t)
    
    return (
        -torch.exp(-z)
        + 0.5 * torch.where(
            2 * t < z,
            torch.exp(-torch.abs(z - t)) * torch.erfc((2 * t - z) / sqrt_4t),
            torch.exp(-z**2 / (4 * t)) * torch.special.erfcx(torch.abs(2 * t - z) / sqrt_4t)
        )
        + 0.5 * torch.exp(-z**2 / (4 * t)) * torch.special.erfcx((2 * t + z) / sqrt_4t)
        - z * torch.erfc(z / sqrt_4t)
        + 2 * torch.sqrt(t / math.pi) * torch.exp(-z**2 / (4 * t))
    )

def W0(z, t):
    safe_t = torch.where(t <= 1e-8, torch.ones_like(t), t)
    return torch.where(t <= 1e-8, torch.exp(-z), U0(z, t) / safe_t)

def z_t_to_s_tau(z, t):
    s = torch.exp(-z*(z+1)/(2*t+z+1))
    tau = 1/(1+t)
    return s, tau

def s_tau_to_z_t(s, tau):
    s_log = torch.log(s)
    t = (1/tau)-1
    z = .5*(-(s_log+1)+torch.sqrt((s_log+1)**2 - 4*s_log*(2*t+1)))
    return z, t    
    
def W_stau(stau):
    z, t = s_tau_to_z_t(stau[:,0], stau[:,1])
    return W0(z,t)



def U_to_W(U, z, t):
    """Apply the W0 rule to an ALREADY-KNOWN U (e.g. interpolated):
       W = U / t, except W = exp(-z) at t ~ 0."""
    safe_t = torch.where(t <= 1e-8, torch.ones_like(t), t)
    return torch.where(t <= 1e-8, torch.exp(-z), U / safe_t)